This code will focus on seeing the results of trying different models for each of the oxides over the entire wavelength range. This models that will be tested are GBR and PLS but not will inckude dimensionality reduction. For a given oxide, the model will only look at the ranges corresponding to thse peaks that we would expect from the indiv oxides.

In [23]:
#libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split,cross_val_predict
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
import openpyxl
import seaborn as sns
from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

In [24]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_seq_items", None)

# numpy: show full arrays
np.set_printoptions(threshold=np.inf, linewidth=np.inf, precision=6, suppress=True)

In [25]:
#file preprocessing:

#opening file
filename='/Users/bigdawg/Library/CloudStorage/Box-Box/LIBS!/Spectral library/Raw Spectra/4:19:2026/zenodo.10K_VACUUM.7.2mJ.csv'
raw=pd.read_csv(filename,low_memory=False)

#renaming to easier columns
wavelength_name=raw.iloc[0,10:].values

raw=raw.drop(index=0)
raw=raw.reset_index(drop=True)

#renaming first row to be the real compunds 
raw.columns=['sample_id', 'SiO2', 'TiO2', 'Al2O3', 'Fe2O3', 'MgO', 'CaO', 'Na2O', 'K2O', 'Pellet Name'] + [float(x) for x in wavelength_name]

ELEMENTS=['SiO2','TiO2', 'Al2O3', 'Fe2O3', 'MgO', 'CaO', 'Na2O', 'K2O']

#creating df of just wavelengths, each row corresponds to same sample
wavelength_cols=raw.columns[10:]
spectra=raw[wavelength_cols].astype(float)

for el in ELEMENTS:
    raw[el]=pd.to_numeric(raw[el],errors='coerce')




print(f'Loaded {len(raw)} spectra with {len(wavelength_cols)} wavelength channels')
print(f'Wavelength range: {wavelength_cols[0]:.2f} to {wavelength_cols[-1]:.2f} nm')
print(f'Available elements: {ELEMENTS}')


Loaded 2503 spectra with 15498 wavelength channels
Wavelength range: 240.86 to 859.41 nm
Available elements: ['SiO2', 'TiO2', 'Al2O3', 'Fe2O3', 'MgO', 'CaO', 'Na2O', 'K2O']


# Removing total weight % outliers

In [26]:
#putting total composition after K2O
raw['total_oxides'] = raw[ELEMENTS].sum(axis=1)
col = raw.pop('total_oxides')
raw.insert(raw.columns.get_loc('K2O') + 1, 'total_oxides', col)
print(raw.columns)

compCols=ELEMENTS + ['total_oxides']


Index([   'sample_id',         'SiO2',         'TiO2',        'Al2O3',
              'Fe2O3',          'MgO',          'CaO',         'Na2O',
                'K2O', 'total_oxides',  'Pellet Name',    240.8587919,
          240.8920957,     240.925399,    240.9587019,    240.9920041,
          241.0253059,    241.0586071,    241.0919078,     241.125208,
          241.1585076,    241.1918067,    241.2251053,    241.2584034,
          241.2917009,    241.3249979,    241.3582943,    241.3915903,
          241.4248857,    241.4581805,    241.4914748,    241.5247686,
          241.5580619,    241.5913546,    241.6246468,    241.6579384,
          241.6912296,    241.7245201,    241.7578102,    241.7910997,
          241.8243886,    241.8576771,    241.8909649,    241.9242523,
          241.9575391,    241.9908254,    242.0241111,    242.0573963,
          242.0906809,     242.123965,    242.1572486,    242.1905316,
           242.223814,     242.257096,    242.2903773,    242.3236582,
      

/var/folders/p1/zrd1bsr50ss60kshx0hm44j80000gn/T/ipykernel_37227/2141590876.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw['total_oxides'] = raw[ELEMENTS].sum(axis=1)
/var/folders/p1/zrd1bsr50ss60kshx0hm44j80000gn/T/ipykernel_37227/2141590876.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw.insert(raw.columns.get_loc('K2O') + 1, 'total_oxides', col)


In [27]:
rawTotComp=raw.copy()
colsToRemove=rawTotComp[rawTotComp['total_oxides']>105].index
print(f'num samples with total oxides >105%: {len(colsToRemove)}')
rawTotComp=rawTotComp.drop(colsToRemove).reset_index(drop=True)

colsToRemove=rawTotComp[rawTotComp['total_oxides']<96].index
print(f'num samples with total oxides <96%: {len(colsToRemove)}')
rawTotComp=rawTotComp.drop(colsToRemove).reset_index(drop=True)



num samples with total oxides >105%: 15
num samples with total oxides <96%: 305


In [28]:

comp_stats = raw[compCols].agg(['mean', 'std']).T

# optional: nicer column names
comp_stats.columns = ['average', 'stdev']

print(f'comp stats with all samples:\n{comp_stats}')


comp_stats = rawTotComp[compCols].agg(['mean', 'std']).T

# optional: nicer column names
comp_stats.columns = ['average', 'stdev']

print(f'comp stats with 96-105 window:\n{comp_stats}')

comp stats with all samples:
                average      stdev
SiO2          57.841746  13.434021
TiO2           1.313035   1.171447
Al2O3         13.558773   4.429121
Fe2O3          8.261901   5.116116
MgO            5.927628   6.864870
CaO            6.251224   5.497922
Na2O           2.716738   1.428350
K2O            2.135417   2.193399
total_oxides  98.006461   5.293868
comp stats with 96-105 window:
                average      stdev
SiO2          58.658893  12.489626
TiO2           1.303084   1.014400
Al2O3         13.710703   3.619425
Fe2O3          8.463426   4.929480
MgO            5.862930   6.511976
CaO            6.253284   4.539447
Na2O           2.877076   1.354556
K2O            2.099979   2.139868
total_oxides  99.229374   0.865171


# Removing based on compositional outliers

In [29]:
df = rawTotComp.copy()

df["is_outlier"]=False
df["outlier_reason"] = ""

In [30]:
avgSiO2 = df["SiO2"].mean()
stdSiO2 = df["SiO2"].std()

threshold=avgSiO2 -2*stdSiO2
print(f'threshold for SiO2 outliers: {threshold:.2f}')
outliers = df[df["SiO2"] < threshold]
print(f'num outliers identified for SiO2: {len(outliers)}')
df.loc[outliers.index, "is_outlier"] = True
df.loc[outliers.index, "outlier_reason"] = "Low SiO2"


threshold for SiO2 outliers: 33.68
num outliers identified for SiO2: 0


In [31]:
threshold=0.01
print(f'threshold for MgO outliers: {threshold:.2f}')
outliers = df[df["MgO"] < threshold]
print(f'num outliers identified for MgO: {len(outliers)}')
df.loc[outliers.index, "is_outlier"] = True
df.loc[outliers.index, "outlier_reason"] = "Low MgO"

threshold for MgO outliers: 0.01
num outliers identified for MgO: 56


In [32]:
avgTiO2 = df["TiO2"].mean()
stdTiO2 = df["TiO2"].std()

threshold=10
print(f'threshold for TiO2 outliers: {threshold:.2f}')
outliers = df[df["TiO2"] > threshold]
print(f'num outliers identified for TiO2: {len(outliers)}')
df.loc[outliers.index, "is_outlier"] = True
df.loc[outliers.index, "outlier_reason"] = "High TiO2"


threshold for TiO2 outliers: 10.00
num outliers identified for TiO2: 0


In [33]:
avgFe2O3 = df["Fe2O3"].mean()
stdFe2O3 = df["Fe2O3"].std()

threshold=20
print(f'threshold for Fe2O3 outliers: {threshold:.2f}')
outliers = df[df["Fe2O3"] > threshold]
print(f'num outliers identified for Fe2O3: {len(outliers)}')
df.loc[outliers.index, "is_outlier"] = True
df.loc[outliers.index, "outlier_reason"] = "High Fe2O3"


threshold for Fe2O3 outliers: 20.00
num outliers identified for Fe2O3: 0


In [34]:
avgCaO = df["CaO"].mean()
stdCaO = df["CaO"].std()

threshold=avgCaO +2.5*stdCaO
print(f'threshold for CaO outliers: {threshold:.2f}')
outliers = df[df["CaO"] > threshold]
print(f'num outliers identified for CaO: {len(outliers)}')
df.loc[outliers.index, "is_outlier"] = True
df.loc[outliers.index, "outlier_reason"] = "High CaO"


threshold for CaO outliers: 17.60
num outliers identified for CaO: 16


In [35]:
avgNa2O = df["Na2O"].mean()
stdNa2O = df["Na2O"].std()

threshold=avgNa2O +2*stdNa2O
print(f'threshold for Na2O outliers: {threshold:.2f}')
outliers = df[df["Na2O"] > threshold]
print(f'num outliers identified for Na2O: {len(outliers)}')
df.loc[outliers.index, "is_outlier"] = True
df.loc[outliers.index, "outlier_reason"] = "High Na2O"


threshold for Na2O outliers: 5.59
num outliers identified for Na2O: 58


Analysis of outliers

In [36]:
numOutliers = df["is_outlier"].sum()
print(f'Number of outliers identified: {numOutliers}')
propOutlier=numOutliers/len(df)
print(f'Proportion of outliers identified: {propOutlier:.2%}')

Number of outliers identified: 130
Proportion of outliers identified: 5.96%


In [37]:
df_no_outlier=df[df['is_outlier']==False]


## Normalizing over wavelength ranges: 
This method of normalization has been extensively tested
beginning with Tucker et al. (2010), K. H. Lepore et al. (2017), Rapin et al. (2017), Thomas et al. (2018), and
especially Ytsma and Dyar (2018). 

In [38]:
#normalization 

def normalize_spectra(spectra):
    UV_cols = [col for col in spectra.columns if 240.81 <= col <= 351.35]
    VIS_cols = [col for col in spectra.columns if 370.16 <= col <= 479.07]
    VNIR_cols = [col for col in spectra.columns if 508.30 <= col <= 859.46]

    totUV = spectra[UV_cols].sum(axis=1).replace(0, np.nan)
    totVIS = spectra[VIS_cols].sum(axis=1).replace(0, np.nan)
    totVNIR = spectra[VNIR_cols].sum(axis=1).replace(0, np.nan)

    spectra_norm = spectra.copy()
    spectra_norm[UV_cols] = spectra[UV_cols].div(totUV, axis=0)
    spectra_norm[VIS_cols] = spectra[VIS_cols].div(totVIS, axis=0)
    spectra_norm[VNIR_cols] = spectra[VNIR_cols].div(totVNIR, axis=0)

    return spectra_norm

In [39]:
df_no_outlier = df[df['is_outlier']==False].reset_index(drop=True)

X = normalize_spectra(df_no_outlier[wavelength_cols].astype(float)).values

Y = df_no_outlier[ELEMENTS].values
groups = df_no_outlier.groupby(ELEMENTS).ngroup().values

print(X.shape, Y.shape, len(np.unique(groups)))

(2053, 15498) (2053, 8) 1964


## Train test split

In [40]:
from sklearn.model_selection import GroupShuffleSplit

TEST_FRACTION = 0.15
N_CANDIDATE_SPLITS = 5000
RANDOM_STATE = 42

# Quantiles used to compare the complete composition distributions
MATCH_QUANTILES = np.array([
    0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95
])

# Scaling prevents high-variance oxides such as SiO2 and MgO
# from dominating the score.
oxide_scale = Y.std(axis=0, ddof=1)
oxide_scale = np.where(oxide_scale > 0, oxide_scale, 1.0)

splitter = GroupShuffleSplit(
    n_splits=N_CANDIDATE_SPLITS,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE
)

best_score = np.inf
best_train_idx = None
best_test_idx = None

for candidate_train_idx, candidate_test_idx in splitter.split(
    X,
    Y,
    groups=groups
):
    Y_candidate_train = Y[candidate_train_idx]
    Y_candidate_test = Y[candidate_test_idx]

    # 1. Difference in means
    mean_loss = np.mean(
        np.abs(
            Y_candidate_train.mean(axis=0)
            - Y_candidate_test.mean(axis=0)
        ) / oxide_scale
    )

    # 2. Difference in standard deviations
    std_loss = np.mean(
        np.abs(
            Y_candidate_train.std(axis=0, ddof=1)
            - Y_candidate_test.std(axis=0, ddof=1)
        ) / oxide_scale
    )

    # 3. Difference across the full distribution
    train_quantiles = np.quantile(
        Y_candidate_train,
        MATCH_QUANTILES,
        axis=0
    )

    test_quantiles = np.quantile(
        Y_candidate_test,
        MATCH_QUANTILES,
        axis=0
    )

    quantile_loss = np.mean(
        np.abs(train_quantiles - test_quantiles)
        / oxide_scale[None, :]
    )

    # 4. Penalize test values outside the training range.
    # Random forests cannot extrapolate beyond training targets.
    below_training = np.maximum(
        Y_candidate_train.min(axis=0)
        - Y_candidate_test.min(axis=0),
        0
    )

    above_training = np.maximum(
        Y_candidate_test.max(axis=0)
        - Y_candidate_train.max(axis=0),
        0
    )

    extrapolation_loss = np.mean(
        (below_training + above_training)
        / oxide_scale
    )

    # 5. Keep the sample percentage close to 15%.
    actual_test_fraction = (
        len(candidate_test_idx) / len(Y)
    )

    size_loss = abs(
        actual_test_fraction - TEST_FRACTION
    )

    total_score = (
        1.0 * mean_loss
        + 0.75 * std_loss
        + 2.0 * quantile_loss
        + 3.0 * extrapolation_loss
        + 5.0 * size_loss
    )

    if total_score < best_score:
        best_score = total_score
        best_train_idx = candidate_train_idx.copy()
        best_test_idx = candidate_test_idx.copy()

In [41]:
train_idx, test_idx = best_train_idx, best_test_idx
X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]
groups_train = groups[train_idx]

print('train:', len(train_idx), ' test:', len(test_idx), f'({len(test_idx)/len(Y):.1%})')
print('groups in both:', len(set(groups[train_idx]) & set(groups[test_idx])))

train: 1747  test: 306 (14.9%)
groups in both: 0


## Models and the 5 split CV, assuming vals for hyperparameters:
I based this on the diffModels one that assumed hyperparameters based on prev papers/standards and got rid of the SVR-lin model.

In [42]:
models = {
    'PLS': PLSRegression(n_components=30, scale=False),
    'GBR': GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, subsample=0.8,
                                     max_features='sqrt', random_state=0),
}

## Finding optimal hyper parameters:
What each tuned parameter controls:
- PLS only has num components so trying all
- GBR n_estimators: number of trees.
- GBR learning_rate: how much each tree counts.
- GBR max_depth: how complex each tree is.
- GBR subsample: the fraction of samples each tree trains on.

In [44]:
searchSpaces = {
    'PLS': {'n_components': list(range(5, 30))},
    'GBR': {'n_estimators': randint(300, 1250),
            'learning_rate': loguniform(0.01, 0.2),
            'max_depth': randint(2, 6),
            'subsample': uniform(0.5, 0.5)},
}

In [45]:
wl = np.array(wavelength_cols, dtype=float)

MASKS = {
    'Si': [(240, 255), (260, 270), (285, 295), (380, 395), (410, 420), (590, 605), (630, 640),
           (695, 705), (765, 780), (790, 800)],
    'Ti': [(250, 255), (280, 285), (300, 355), (370, 380), (385, 395), (425, 435), (450, 460),
           (505, 540), (545, 555), (560, 615), (620, 630), (650, 660), (715, 730), (840, 850)],
    'Al': [(250, 270), (275, 285), (305, 315), (390, 400), (555, 565), (620, 630), (680, 695),
           (700, 715), (730, 740), (745, 750), (780, 790)],
    'Fe': [(240, 280), (370, 445), (505, 565), (635, 645), (715, 725), (745, 755), (820, 825)],
    'Mg': [(275, 290), (380, 390), (445, 455), (515, 525), (550, 555), (785, 795)],
    'Ca': [(310, 325), (390, 400), (420, 425), (845, 860)],
    'Na': [(325, 335), (430, 455), (460, 470), (585, 595), (815, 825)],
    'K':  [(300, 350), (400, 410), (760, 775)],
}
OXIDE_ELEMENT = {'SiO2': 'Si', 'TiO2': 'Ti', 'Al2O3': 'Al', 'Fe2O3': 'Fe',
                 'MgO': 'Mg', 'CaO': 'Ca', 'Na2O': 'Na', 'K2O': 'K'}

inMask = {}
for oxide, el in OXIDE_ELEMENT.items():
    keep = np.zeros(len(wl), dtype=bool)
    for lo, hi in MASKS[el]:
        keep = keep | ((wl >= lo) & (wl <= hi))
    inMask[oxide] = keep
    print(oxide, keep.sum(), 'channels')


combined = np.zeros(len(wl), dtype=bool)
for oxide in OXIDE_ELEMENT:
    combined = combined | inMask[oxide]
print('combined mask:', combined.sum(), 'channels')

SiO2 3309 channels
TiO2 6265 channels
Al2O3 3322 channels
Fe2O3 5626 channels
MgO 1688 channels
CaO 1429 channels
Na2O 2015 channels
K2O 2306 channels
combined mask: 13176 channels


In [46]:
import subprocess, os
subprocess.Popen(['caffeinate', '-i', '-w', str(os.getpid())])

if os.path.exists('maskResults.pkl'):
    bestParams, results = pd.read_pickle('maskResults.pkl')
else:
    bestParams, results = {}, []
done = {(r['features'], r['model'], r['oxide']) for r in results}

for featureSet in ['mask', 'combined', 'full']:
    for name in ['PLS', 'GBR']:
        for j, oxide in enumerate(ELEMENTS):
            if (featureSet, name, oxide) in done:
                continue

            if featureSet == 'mask':
                cols = inMask[oxide]
            elif featureSet == 'combined':
                cols = combined
            else:
                cols = np.ones(X.shape[1], dtype=bool)
            
            Xtr, Xte = X_train[:, cols], X_test[:, cols]
            mean, std = Y_train[:, j].mean(), Y_train[:, j].std()

            search = RandomizedSearchCV(models[name], searchSpaces[name], n_iter=60,
                                        cv=GroupKFold(n_splits=5, shuffle=True, random_state=42),
                                        scoring='neg_root_mean_squared_error',
                                        n_jobs=-1, random_state=0)
            search.fit(Xtr, (Y_train[:, j] - mean) / std, groups=groups_train)

            predTrain = search.predict(Xtr).ravel() * std + mean
            predTest = search.predict(Xte).ravel() * std + mean

            bestParams[(featureSet, name, oxide)] = search.best_params_
            results.append({'features': featureSet, 'model': name, 'oxide': oxide,
                            'n_channels': int(cols.sum()),
                            'RMSE-CV': -search.best_score_ * std,
                            'RMSE-C': np.sqrt(np.mean((Y_train[:, j] - predTrain) ** 2)),
                            'predTest': predTest})
            pd.to_pickle((bestParams, results), 'maskResults.pkl')
            print(featureSet, name, oxide, int(cols.sum()), 'channels  RMSE-CV:',
                  round(results[-1]['RMSE-CV'], 3), search.best_params_)

/opt/anaconda3/envs/libs/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 25 is smaller than n_iter=60. Running 25 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


mask PLS SiO2 3309 channels  RMSE-CV: 3.279 {'n_components': 28}


/opt/anaconda3/envs/libs/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 25 is smaller than n_iter=60. Running 25 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/opt/anaconda3/envs/libs/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


mask PLS TiO2 6265 channels  RMSE-CV: 0.365 {'n_components': 29}


/opt/anaconda3/envs/libs/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 25 is smaller than n_iter=60. Running 25 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/opt/anaconda3/envs/libs/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


...

full PLS CaO 15498 channels  RMSE-CV: 1.042 {'n_components': 29}


/opt/anaconda3/envs/libs/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 25 is smaller than n_iter=60. Running 25 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/opt/anaconda3/envs/libs/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


full PLS Na2O 15498 channels  RMSE-CV: 0.449 {'n_components': 22}


/opt/anaconda3/envs/libs/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 25 is smaller than n_iter=60. Running 25 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/opt/anaconda3/envs/libs/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


full PLS K2O 15498 channels  RMSE-CV: 0.636 {'n_components': 29}
full GBR SiO2 15498 channels  RMSE-CV: 2.799 {'learning_rate': np.float64(0.022453728765539232), 'max_depth': 4, 'n_estimators': 1175, 'subsample': np.float64(0.9810942725587191)}
full GBR TiO2 15498 channels  RMSE-CV: 0.304 {'learning_rate': np.float64(0.022453728765539232), 'max_depth': 4, 'n_estimators': 1175, 'subsample': np.float64(0.9810942725587191)}
full GBR Al2O3 15498 channels  RMSE-CV: 1.626 {'learning_rate': np.float64(0.04147791158458682), 'max_depth': 4, 'n_estimators': 769, 'subsample': np.float64(0.8580372656143216)}
full GBR Fe2O3 15498 channels  RMSE-CV: 1.2 {'learning_rate': np.float64(0.015647203367978108), 'max_depth': 5, 'n_estimators': 788, 'subsample': np.float64(0.5812464673381874)}
full GBR MgO 15498 channels  RMSE-CV: 1.34 {'learning_rate': np.float64(0.017311609891130626), 'max_depth': 4, 'n_estimators': 671, 'subsample': np.float64(0.5100537730937468)}
full GBR CaO 15498 channels  RMSE-CV: 0.8

In [47]:
res = pd.DataFrame(results)

for name in ['PLS', 'GBR']:
    t = res[res.model == name].pivot(index='oxide', columns='features', values='RMSE-CV').loc[ELEMENTS, ['full', 'combined', 'mask']]
    t['combined vs full %'] = 100 * (t['combined'] - t['full']) / t['full']
    t['mask vs full %'] = 100 * (t['mask'] - t['full']) / t['full']
    t['mask channels'] = [inMask[ox].sum() for ox in ELEMENTS]
    print(f'\n{name} RMSE-CV (wt%), training data only  (combined mask = {combined.sum()} channels)')
    display(t.round(3))


PLS RMSE-CV (wt%), training data only  (combined mask = 13176 channels)


features,full,combined,mask,combined vs full %,mask vs full %,mask channels
oxide,,,,,,
SiO2,3.214,3.219,3.279,0.141,2.022,3309
TiO2,0.351,0.352,0.365,0.110,3.990,6265
Al2O3,1.621,1.622,1.639,0.080,1.101,3322
Fe2O3,1.491,1.492,1.526,0.105,2.332,5626
MgO,1.687,1.686,1.651,-0.083,-2.103,1688
CaO,1.042,1.040,1.060,-0.187,1.785,1429
Na2O,0.449,0.450,0.498,0.159,10.885,2015
K2O,0.636,0.636,0.681,-0.091,7.087,2306



GBR RMSE-CV (wt%), training data only  (combined mask = 13176 channels)


features,full,combined,mask,combined vs full %,mask vs full %,mask channels
oxide,,,,,,
SiO2,2.799,2.789,2.928,-0.345,4.604,3309
TiO2,0.304,0.303,0.311,-0.464,2.248,6265
Al2O3,1.626,1.614,1.639,-0.711,0.792,3322
Fe2O3,1.200,1.197,1.202,-0.241,0.152,5626
MgO,1.340,1.344,1.388,0.237,3.588,1688
CaO,0.859,0.848,0.907,-1.376,5.482,1429
Na2O,0.422,0.423,0.447,0.369,6.114,2015
K2O,0.616,0.617,0.669,0.115,8.487,2306


In [48]:
rng = np.random.default_rng(0)
boot = rng.integers(0, len(Y_test), size=(2000, len(Y_test)))
rows = []

for name in ['PLS', 'GBR']:
    for j, oxide in enumerate(ELEMENTS):
        sel = (res.model == name) & (res.oxide == oxide)
        predFull = res[sel & (res.features == 'full')]['predTest'].iloc[0]
        errFull = (Y_test[:, j] - predFull) ** 2
        row = {'model': name, 'oxide': oxide, 'full RMSE-P': np.sqrt(errFull.mean())}

        for featureSet in ['combined', 'mask']:
            pred = res[sel & (res.features == featureSet)]['predTest'].iloc[0]
            err = (Y_test[:, j] - pred) ** 2
            gap = np.sqrt(errFull[boot].mean(axis=1)) - np.sqrt(err[boot].mean(axis=1))
            lo, hi = np.percentile(gap, [2.5, 97.5])
            row[f'{featureSet} RMSE-P'] = np.sqrt(err.mean())
            row[f'{featureSet} vs full 95% CI'] = f'{lo:.3f} to {hi:.3f}'
        rows.append(row)

display(pd.DataFrame(rows).round(3))

,model,oxide,full RMSE-P,combined RMSE-P,combined vs full 95% CI,mask RMSE-P,mask vs full 95% CI
0,PLS,SiO2,2.755,2.755,-0.008 to 0.008,3.014,-0.475 to -0.050
1,PLS,TiO2,0.374,0.374,-0.002 to 0.001,0.383,-0.026 to 0.009
2,PLS,Al2O3,1.373,1.390,-0.032 to -0.004,1.503,-0.230 to -0.031
3,PLS,Fe2O3,1.380,1.383,-0.008 to 0.002,1.385,-0.060 to 0.053
4,PLS,MgO,1.456,1.458,-0.008 to 0.003,1.503,-0.122 to 0.028
5,PLS,CaO,1.044,1.062,-0.037 to 0.003,1.087,-0.113 to 0.040
6,PLS,Na2O,0.424,0.425,-0.006 to 0.002,0.454,-0.059 to -0.002
7,PLS,K2O,0.568,0.567,-0.006 to 0.009,0.630,-0.110 to -0.009
8,GBR,SiO2,2.365,2.410,-0.121 to 0.033,2.564,-0.345 to -0.059
9,GBR,TiO2,0.332,0.343,-0.025 to 0.002,0.346,-0.027 to 0.000
